# Clustering Energy Analysis

Thin runner notebook for LRK/xKV clustering comparisons across keys and values, with independent cells for each tensor and clustering axis.

In [42]:
from pathlib import Path
import sys

import torch


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "utils").is_dir():
            return path
    raise RuntimeError("Could not find repo root containing ./utils")


repo_root = find_repo_root(Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from analysis.utils import display_case_result, load_kv_dump, run_analysis_case


In [44]:
kv_dump_path = (
    repo_root
    / "results/kv_dumps/meta-llama_Llama-3.1-8B-Instruct/niah_multiquery_raw_kv.pt"
)
loaded = load_kv_dump(kv_dump_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

raw_keys = loaded["raw_keys"]
keys = loaded["keys"].to(device)
values = loaded["values"].to(device)
prompt_len = loaded["prompt_len"]
rope_theta = loaded["rope_theta"]

keys.shape, values.shape, prompt_len, rope_theta


Using device: cuda


(torch.Size([32, 1, 8, 3770, 128]),
 torch.Size([32, 1, 8, 3770, 128]),
 3770,
 500000.0)

In [48]:
show_detail_tables = False

kmeans_cfg = {
    "n_clusters": 3,
    "kmeans_cluster_size": 512,
    "kmeans_n_iter": 16,
    "kmeans_init": "kmeans++",
    "kmeans_dtype": torch.float32,
}

decomposition_cfg = {
    "decomposition_method": "svd",
    "rank_selection": "comp_ratio",
    "comp_ratio": 1.5,
    "energy_threshold": 0.95,
    "decomp_n_iter": 3,
    "decomp_lr": 1e-2,
}

lrk_mode = "per_head"  # "avg_heads" or "per_head"
include_lrk_head_breakdown = False
prefix_end = prompt_len
local_window = 0

xkv_layer_group_size = 4
xkv_num_layers = keys.size(0)

## Keys, Row Clustering

In [49]:
keys_rows = run_analysis_case(
    keys,
    tensor_name="keys",
    cluster_axis="rows",
    kmeans_cfg=kmeans_cfg,
    decomposition_cfg=decomposition_cfg,
    lrk_mode=lrk_mode,
    include_lrk_head_breakdown=include_lrk_head_breakdown,
    prefix_end=prefix_end,
    local_window=local_window,
    xkv_layer_group_size=xkv_layer_group_size,
    xkv_num_layers=xkv_num_layers,
)

display_case_result(keys_rows, show_detail_tables=show_detail_tables)


,case,rows,mean_eta,median_eta,mean_relative_low_rank_recon_error,median_relative_low_rank_recon_error,min_eta,max_eta
0,keys_lrk_rows,256,0.232446,0.221463,0.096696,0.102828,0.034217,0.480861
1,keys_xkv_rows,40,0.237655,0.242647,0.066549,0.068880,0.114890,0.375000
2,keys_lrk_rows_n_clusters_1,256,0.315142,0.304667,0.094801,0.101043,0.074367,0.558442
3,keys_xkv_rows_n_clusters_1,40,0.311980,0.306936,0.050022,0.052382,0.173713,0.473684


## Values, Row Clustering

In [50]:
values_rows = run_analysis_case(
    values,
    tensor_name="values",
    cluster_axis="rows",
    kmeans_cfg=kmeans_cfg,
    decomposition_cfg=decomposition_cfg,
    lrk_mode=lrk_mode,
    include_lrk_head_breakdown=include_lrk_head_breakdown,
    prefix_end=prefix_end,
    local_window=local_window,
    xkv_layer_group_size=xkv_layer_group_size,
    xkv_num_layers=xkv_num_layers,
)

display_case_result(values_rows, show_detail_tables=show_detail_tables)


/mnt/jfs-00/team-agent/home/m84366023/gist_vs_details/utils/matrix_decomposition.py:40: UserWarning: Target compression ratio 1.5 is too high for matrix of shape (1, 128). Using rank 1.
  warnings.warn(


,case,rows,mean_eta,median_eta,mean_relative_low_rank_recon_error,median_relative_low_rank_recon_error,min_eta,max_eta
0,values_lrk_rows,256,0.786595,0.812633,0.313707,0.316987,0.142857,0.953368
1,values_xkv_rows,40,0.814260,0.831621,0.196665,0.195783,0.466837,0.888889
2,values_lrk_rows_n_clusters_1,256,0.891137,0.922806,0.347952,0.352049,0.213010,0.988571
3,values_xkv_rows_n_clusters_1,40,0.883943,0.905339,0.158964,0.156785,0.528061,0.944751


## Keys, Column Clustering

In [6]:
keys_cols = run_analysis_case(
    keys,
    tensor_name="keys",
    cluster_axis="cols",
    kmeans_cfg=kmeans_cfg,
    decomposition_cfg=decomposition_cfg,
    lrk_mode=lrk_mode,
    include_lrk_head_breakdown=include_lrk_head_breakdown,
    prefix_end=prefix_end,
    local_window=local_window,
    xkv_layer_group_size=xkv_layer_group_size,
    xkv_num_layers=xkv_num_layers,
)

display_case_result(keys_cols, show_detail_tables=show_detail_tables)


/mnt/jfs-00/team-agent/home/m84366023/gist_vs_details/utils/matrix_decomposition.py:40: UserWarning: Target compression ratio 16.0 is too high for matrix of shape (1, 3770). Using rank 1.
  warnings.warn(
/mnt/jfs-00/team-agent/home/m84366023/gist_vs_details/utils/matrix_decomposition.py:40: UserWarning: Target compression ratio 16.0 is too high for matrix of shape (12, 3770). Using rank 1.
  warnings.warn(
/mnt/jfs-00/team-agent/home/m84366023/gist_vs_details/utils/matrix_decomposition.py:40: UserWarning: Target compression ratio 16.0 is too high for matrix of shape (7, 3770). Using rank 1.
  warnings.warn(
/mnt/jfs-00/team-agent/home/m84366023/gist_vs_details/utils/matrix_decomposition.py:40: UserWarning: Target compression ratio 16.0 is too high for matrix of shape (15, 3770). Using rank 1.
  warnings.warn(
/mnt/jfs-00/team-agent/home/m84366023/gist_vs_details/utils/matrix_decomposition.py:40: UserWarning: Target compression ratio 16.0 is too high for matrix of shape (16, 3770). Usi

,case,rows,mean_eta,median_eta,mean_relative_low_rank_recon_error,median_relative_low_rank_recon_error,min_eta,max_eta
0,keys_lrk_cols,256,0.531487,0.520102,0.458373,0.460161,0.279070,0.815436
1,keys_xkv_cols,40,0.586010,0.580553,0.246824,0.250494,0.489655,0.738806
2,keys_lrk_cols_n_clusters_1,256,0.992535,0.993506,0.403912,0.405231,0.932039,1.000000
3,keys_xkv_cols_n_clusters_1,40,0.998950,1.000000,0.220335,0.225099,0.992481,1.000000


## Values, Column Clustering

In [7]:
values_cols = run_analysis_case(
    values,
    tensor_name="values",
    cluster_axis="cols",
    kmeans_cfg=kmeans_cfg,
    decomposition_cfg=decomposition_cfg,
    lrk_mode=lrk_mode,
    include_lrk_head_breakdown=include_lrk_head_breakdown,
    prefix_end=prefix_end,
    local_window=local_window,
    xkv_layer_group_size=xkv_layer_group_size,
    xkv_num_layers=xkv_num_layers,
)

display_case_result(values_cols, show_detail_tables=show_detail_tables)


/mnt/jfs-00/team-agent/home/m84366023/gist_vs_details/utils/matrix_decomposition.py:40: UserWarning: Target compression ratio 16.0 is too high for matrix of shape (8, 3770). Using rank 1.
  warnings.warn(
/mnt/jfs-00/team-agent/home/m84366023/gist_vs_details/utils/matrix_decomposition.py:40: UserWarning: Target compression ratio 16.0 is too high for matrix of shape (13, 3770). Using rank 1.
  warnings.warn(
/mnt/jfs-00/team-agent/home/m84366023/gist_vs_details/utils/matrix_decomposition.py:40: UserWarning: Target compression ratio 16.0 is too high for matrix of shape (4, 3770). Using rank 1.
  warnings.warn(
/mnt/jfs-00/team-agent/home/m84366023/gist_vs_details/utils/matrix_decomposition.py:40: UserWarning: Target compression ratio 16.0 is too high for matrix of shape (5, 3770). Using rank 1.
  warnings.warn(
/mnt/jfs-00/team-agent/home/m84366023/gist_vs_details/utils/matrix_decomposition.py:40: UserWarning: Target compression ratio 16.0 is too high for matrix of shape (3, 3770). Using

,case,rows,mean_eta,median_eta,mean_relative_low_rank_recon_error,median_relative_low_rank_recon_error,min_eta,max_eta
0,values_lrk_cols,256,0.901338,0.929742,0.859295,0.873704,0.382653,0.972332
1,values_xkv_cols,40,0.938044,0.947286,0.650989,0.652785,0.755102,0.975610
2,values_lrk_cols_n_clusters_1,256,0.991860,0.992308,0.828255,0.845233,0.962963,1.000000
3,values_xkv_cols_n_clusters_1,40,0.999150,1.000000,0.600733,0.598437,0.993827,1.000000
